In [ ]:
#Specific grain boundary function
def grain_boundaries(GB_file_path, GI_file_path, pixel_index, save_path, GB_colour='input', GI_colour='input'):
    """
    Fits Gaussian to GB and GI data, plots the results, saves the plot, 
    and writes the analysis data to a text file.
    """
    # Pixel number corresponds to the 'pixels' value (1 to 5)
    pixels = pixel_index 
    #Plotting parameters
    figsize_x, figsize_y = 3.5, 2.5 
    tick_length = 4 
    legend_font = 10 
    scatter_size = 15 
    base_font_size = 10
    line_width = 1.5
    # Perform fitting
    GB_x_raw_1, GB_y_raw_1, GB_x_1, GB_y_1, GB_peak_1, GB_fwhm_1 = fit_gaussian(GB_file_path, rows)
    GI_x_raw_1, GI_y_raw_1, GI_x_1, GI_y_1, GI_peak_1, GI_fwhm_1 = fit_gaussian(GI_file_path, rows)
    peak_diff_1 = GI_peak_1 - GB_peak_1

    # Plotting functions: 
    fig, ax = plt.subplots(1, 1, sharey=True, figsize=(figsize_x, figsize_y))
    plt.rcParams.update({'font.size': base_font_size}) # Use a slightly smaller font for general text
    
    ax.tick_params(direction='in', length=tick_length, labelsize=base_font_size)
    ax.tick_params(axis='both')#, labelsize=18)
    ax.margins(0.05, 0.05) # Reduced margins for a tighter plot
    ax.set_xlabel(f"CPD (mV)")#, fontsize=22)
    
    # Calculate limits from all fitted data for consistent scaling
    x_data_all = np.concatenate([GB_x_raw_1, GB_x_1, GI_x_raw_1, GI_x_1])
    ax.set_xlim(np.min(x_data_all) - 0.01, np.max(x_data_all) + 0.01)
    ax.set_ylabel("Distribution (V$^{-1}$)")#, fontsize=22)
    
    # Limit number of ticks
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5)) 
    
    # Plotting
    plt.scatter(GB_x_raw_1, GB_y_raw_1, label=f"GB", color=GB_colour, s=scatter_size)
    plt.plot(GB_x_1, GB_y_1, label=f"_nolegend_", color=GB_colour, linewidth=line_width)
    plt.scatter(GI_x_raw_1, GI_y_raw_1, label=f"GI", color=GI_colour, s=scatter_size, marker='s')
    plt.plot(GI_x_1, GI_y_1, label=f"_nolegend_", color=GI_colour, linewidth=line_width, linestyle='--')
    
    plt.legend(loc='upper right', fontsize=legend_font)
    
    # Save Plot
    # Assumes save_plot_to_folder is defined and handles the saving
    fig=plt.gcf()
    plot_filename = f'GrainBoundary_Interior_MultiGaussian_{pixels*20}_nm.jpeg'
    save_plot_to_folder(fig, save_path, filename=plot_filename)
    plt.close(fig) # Close figure to prevent memory issues

    # Save Analysis Output
    output_file = os.path.join(save_path, f'GrainBoundaries_analysis_output_{pixels*20}_nm.txt')
    with open(output_file, 'w') as f:
        output_lines = [
            f"Analysis for {pixels*20} nm:",
            "Grain boundary peak = " + str(GB_peak_1),
            "Grain boundary fwhm = " + str(GB_fwhm_1),
            "Grain interior peak = " + str(GI_peak_1),
            "Grain interior fwhm = " + str(GI_fwhm_1),
            "Diff of GB/GI =" + str(peak_diff_1),
        ]
        for line in output_lines:
            f.write(line + '\n')
            
    return peak_diff_1, GB_peak_1, GI_peak_1